In [ ]:
import jax 
import jax.numpy as jnp 

from functools import partial

jax.config.update("jax_enable_x64", True)

# Actual transport operator

First, we use a single-angle transport operator:

$$ T_{gk}$$

We setup a cartesian block domain using second order tetrahedral elements

In [ ]:
import jax_sn
from jax_sn.domain import CrossSectionData, Domain, TopologicalSortedAngle
from jax_sn.solution_domain import BasixLagrangianSimplex, SolutionDomain
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.operators.mass_operator import MassOperator
from jax_sn.operators.mass_operator.mass_operator_base import _mass_kernel_idx_in_precomp_vmap
from jax_sn.operators.transport_operator import TransportOperatorBatched, transport_inverse_sweep_batched, TransportOperatorVmap, TransportOperatorBatchedPallas, TransportOperatorVmapPallas
import pyvista as pv
from jax_sbgeom.jax_utils.mesh import mesh_to_pyvista_mesh
import equinox as eqx
import numpy as onp

def setup_base_case(dim, degree):
    n_elem       = 100
    n_other_dim  = 10
    other_dims   = [[(1.0, n_other_dim)]]*(dim - 1)
    xs0          = CrossSectionData.constant(0.2, 1, 1)    
    verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)]] + other_dims, [xs0], [1.0], 0)
    element_type                         = BasixLagrangianSimplex(degree, dim)
    domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections)    
    solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
    sources = jnp.repeat(sources, repeats = solution_domain.n_basis, axis = -1).reshape(-1, solution_domain.n_basis)
    return solution_domain, sources


solution_domain, sources = setup_base_case(3,2)
i_tn     = 50
angle_i  = jax_sn.quadrature_set.create_tn_quadrature_set(4)[i_tn]
toposort = TopologicalSortedAngle.from_domain_angle(solution_domain.domain, angle_i[0], 100)
x_shape  = jnp.ones((solution_domain.domain.n_elements, solution_domain.n_basis))



Then, we parametrise the sweep by for simplicity, modifying the total cross section.

In other words, $\theta = \sigma$, and we have 

$S(theta, x) = u = T^{-1}(\theta) \circ H[x]$

Since the 'creation' of the T operator is nothing more than pointing a few arrays to some precomputed arrays, this does not have any overhead. We perform some PyTree surgery to obtain a new solution domain with the updated total cross sections and then 'create' the T operator.

Closing over the sigma parameter, we have the function that provides the above equation.

A simple objective function is just the total 'absorption' (that I pick because it depends both on $\theta$ directly and implicitly!):

$$ f(\theta, u) = \int \sigma(\theta) u dV $$

In [ ]:

def get_modified_solution_domain_data(sigma, solution_domain_data):
    old_total = solution_domain_data.domain.cross_sections_indexed.cross_section_data.total
    new_total = old_total.at[0,0].set(sigma)
    return eqx.tree_at(lambda solution_domain_data : solution_domain_data.domain.cross_sections_indexed.cross_section_data.total, solution_domain_data, new_total)    


def get_transport_operator(sigma, solution_domain_data, toposort):    
    
    top = TransportOperatorBatchedPallas.from_solution_domain_data_and_toposort(get_modified_solution_domain_data(sigma, solution_domain_data), toposort, batch_size = 1024)
    return top
mass_operator   = MassOperator.from_fem_matrices_and_precomputed_geometric_data(solution_domain.fem_matrices, solution_domain.precomputed_geometric_data)
get_top_closure = partial(get_transport_operator, solution_domain_data=solution_domain.solution_domain_data, toposort=toposort)
x_base          = jnp.ones(x_shape.shape)

top = get_top_closure(0.2)
result_closure  = jax.jit(lambda t : get_top_closure(t).inv(mass_operator(x_base), 0))

In [ ]:
from jax_sn.operators.transport_operator.transport_operator_forward import forward_transport_operator_batch, forward_transport_operator_vmap
from jax_sn.operators.transport_operator.transport_operator_imp import transport_inverse_sweep_batched, transport_inverse_sweep_complete_vmap


In [ ]:
@partial(jax.jit, static_argnums = 2)
def stupid_matvec(amat, v, batch_size : int):
    return amat @ v + jnp.arange(batch_size).sum()

amat = jnp.ones((10,10))
v   = jnp.ones((10,))

stupid_matvec(amat, v, 1024)

In [ ]:
def okaywtf(top):
    return jax.lax.custom_linear_solve(lambda v : stupid_matvec(amat, v, top.batch_size), b = v, solve = lambda _, v : amat @ v, transpose_solve= lambda _, v : amat.T @ v)

okaywtf(top)

In [ ]:
top.inv(x_base,0)

In [ ]:
quad_order = 2 * solution_domain.element.element.degree
quad_points, quad_weights = solution_domain.element.volume_quadrature_order(quad_order)    
#@jax.jit
def integrate_sigma_u(solution_domain_data, solution):
    total_xs = solution_domain_data.domain.cross_sections_indexed.map_to_full_array(0)
    def _integrate_elem(elem_idx : int) -> float:
        elem_tot = jnp.full(quad_points.shape[0], elem_idx, dtype=int)        
        numerical_values  = solution_domain.element.interpolate_from_local_coordinate(quad_points, elem_tot, solution)        
        return jnp.sum(quad_weights * numerical_values )* solution_domain.precomputed_geometric_data.vol_jacobians_det[elem_idx] * total_xs.total[elem_idx]    
    return jax.vmap(_integrate_elem)(jnp.arange(solution_domain.domain.n_elements))

sigma_u_int = integrate_sigma_u(solution_domain.solution_domain_data, result_closure(0.2))
@jax.jit
def total_closure(sigma, solution_domain_data, toposort):
    mod_solution_domain_data = get_modified_solution_domain_data(sigma, solution_domain_data)
    top                      = TransportOperatorVmapPallas(mod_solution_domain_data, toposort)#, 1000)#.inv(mass_operator(x_base), 0)
    #print(top.inv)
    
    result = top.inv(mass_operator(x_base), 0)
    return jnp.sum(integrate_sigma_u(solution_domain_data, result))

total_closure(0.2, solution_domain.solution_domain_data, toposort).block_until_ready()

In [ ]:
#%timeit total_closure(0.2, solution_domain.solution_domain_data, toposort).block_until_ready()

In [ ]:
%timeit total_closure(0.2, solution_domain.solution_domain_data, toposort   ).block_until_ready()

In [ ]:
try:
    f_grad = jax.jit(jax.grad(total_closure, argnums=0))#(0.2, solution_domain.solution_domain_data, toposort)
except NotImplementedError as e:
    print("NotImplementedError:", e)

In [ ]:
%timeit f_grad(0.2, solution_domain.solution_domain_data, toposort).block_until_ready()

In [ ]:
a = total_closure(0.2, solution_domain.solution_domain_data, toposort)

In [ ]:
b = total_closure(0.2 + 1e-3, solution_domain.solution_domain_data, toposort)

In [ ]:
stepsize = 1e-8
a = total_closure(0.2, solution_domain.solution_domain_data, toposort)
b = total_closure(0.2 + stepsize, solution_domain.solution_domain_data, toposort)
print((b-a) / stepsize)

In [ ]:
f_grad(0.2, solution_domain.solution_domain_data, toposort).block_until_ready()

In [ ]:
plot_solution = False
if plot_solution:
    result = solution_domain.interpolate(solution_domain.domain.vertices, result_closure(0.2)) ## this takes a while because the compilation of the BVH building and tracing is quite complicated...
    plotter = pv.Plotter()
    plotter.add_mesh(mesh_to_pyvista_mesh((solution_domain.domain.vertices, solution_domain.domain.connectivity)), scalars = sigma_u_int, show_edges=False)
    plotter.add_arrows(onp.array([0.0,0.0,0.0]), angle_i[0], mag=2.0, color="red")
    plotter.show()


In [ ]:
plot_solution = False
if plot_solution:
    result = solution_domain.interpolate(solution_domain.domain.vertices, result_closure(0.2)) ## this takes a while because the compilation of the BVH building and tracing is quite complicated...
    plotter = pv.Plotter()
    plotter.add_mesh(mesh_to_pyvista_mesh((solution_domain.domain.vertices, solution_domain.domain.connectivity)), scalars = result, show_edges=False)
    plotter.add_arrows(onp.array([0.0,0.0,0.0]), angle_i[0], mag=2.0, color="red")
    plotter.show()
